# TMDB Feature Engineering

In [1]:
import pandas as pd
import numpy as np

tmdb_df = pd.read_csv("../data/tmdb_movies.csv")



### `Genre` - Multi-hot Encoding


In [2]:
genre_features = tmdb_df["genres"].str.get_dummies(sep="|")

print("Number of genre features:", genre_features.shape[1])
print("Genres:", genre_features.columns.tolist())



Number of genre features: 19
Genres: ['Action', 'Adventure', 'Animation', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Family', 'Fantasy', 'History', 'Horror', 'Music', 'Mystery', 'Romance', 'Science Fiction', 'TV Movie', 'Thriller', 'War', 'Western']


### `runtime` scaling

Note: Runtime is temporarily standardised using the TMDB Dataset for feature engineering. The scaler will be refitted using the training data only after merging with the recommendation dataset.

In [3]:
from sklearn.preprocessing import StandardScaler

runtime_scaler = StandardScaler()

runtime_feature = runtime_scaler.fit_transform(
    tmdb_df[["runtime"]]
)

runtime_feature[:5]

array([[-0.19957291],
       [ 0.23259712],
       [ 0.17622712],
       [ 0.66476715],
       [ 0.27017713]])

### `original_language` One-hot

In [4]:
language_counts = tmdb_df["original_language"].value_counts()

print(language_counts.to_string())

# change threshold to 1000 test
print(language_counts[language_counts > 1000].sum())
print(language_counts[language_counts < 1000].sum())

original_language
en    54050
fr     5166
it     3330
ja     2949
es     2872
de     2205
ru     2144
hi     1232
zh     1046
ko     1044
pt      882
sv      853
cn      750
fi      661
pl      658
nl      590
tr      449
ta      427
da      387
cs      362
fa      291
no      279
el      275
te      262
ml      241
hu      238
he      228
ar      214
th      206
tl      160
xx      149
ro      147
uk      115
bn      105
et       98
id       89
sr       84
is       79
ka       55
mr       55
hr       53
sh       48
ca       47
vi       41
lv       39
sk       37
lt       35
bg       33
kn       30
ms       27
sl       26
ur       25
hy       22
kk       22
bs       19
eu       19
sq       19
ku       16
pa       15
mk       14
az       14
af       13
gu       11
wo       10
ne        9
gl        9
bo        8
uz        8
ky        8
sw        8
mn        7
am        7
ps        5
km        5
iu        4
bm        4
dz        4
zu        4
ln        4
lo        4
lb        4
si        

Freq < 100 will be considered as `others` (test 1000)

In [5]:
language_counts = tmdb_df["original_language"].value_counts()

tmdb_df["language_grouped"] = tmdb_df["original_language"].where(
    tmdb_df["original_language"].map(language_counts) >= 1000,
    "Other"
)

language_features = pd.get_dummies(
    tmdb_df["language_grouped"],
    prefix="lang",
    dtype=int
)

print("Number of language features:", language_features.shape[1])
print(language_features.columns.tolist())

Number of language features: 11
['lang_Other', 'lang_de', 'lang_en', 'lang_es', 'lang_fr', 'lang_hi', 'lang_it', 'lang_ja', 'lang_ko', 'lang_ru', 'lang_zh']


### `Origin Country` Multi-Hot

In [6]:
country_counts = (
    tmdb_df["origin_country"]
    .str.split("|")
    .explode()
    .value_counts()
)

print(country_counts.to_string())
print(country_counts[country_counts < 1000].sum())
print(country_counts[country_counts > 1000].sum())


origin_country
US    48719
GB     7282
FR     5658
CA     3799
IT     3698
JP     3040
DE     2528
IN     2384
ES     1905
RU     1207
HK     1187
AU     1125
KR     1061
SU     1050
SE      928
MX      806
BR      787
CN      751
FI      725
PL      704
DK      674
NL      640
CH      512
TR      470
IE      454
AR      438
AT      392
BE      380
NO      333
IR      299
GR      299
TW      268
IL      266
HU      260
NZ      230
TH      218
CZ      207
XC      203
PT      201
PH      185
RO      174
UA      142
ZA      106
ID      102
IS       95
CL       95
EE       95
YU       91
EG       90
SG       82
CO       66
RS       65
BG       63
PE       61
HR       54
LU       49
SA       45
KZ       44
LV       44
MY       43
VN       42
LT       42
BD       42
AE       39
XG       36
GE       32
MA       31
SI       30
UY       29
CU       28
VE       28
NG       28
LB       27
BA       26
SK       25
TN       22
SN       22
PK       22
PS       20
MK       18
BY       18
QA       16
A

Countries count < 100 will be considered as others (test 1000)

In [7]:
# Countries appearing at least 1000 (test) times
common_countries = country_counts[country_counts >= 1000].index

# Create multi-hot encoding
country_features = (
    tmdb_df["origin_country"]
    .str.get_dummies(sep="|")
)

# Keep common countries only
country_features = country_features[
    [c for c in country_features.columns if c in common_countries]
]

# Add an Other column for movies with no common country
country_features["Other"] = (country_features.sum(axis=1) == 0).astype(int)

# Add prefix
country_features = country_features.add_prefix("country_")

print("Number of country features:", country_features.shape[1])
print(country_features.columns.tolist())

Number of country features: 15
['country_AU', 'country_CA', 'country_DE', 'country_ES', 'country_FR', 'country_GB', 'country_HK', 'country_IN', 'country_IT', 'country_JP', 'country_KR', 'country_RU', 'country_SU', 'country_US', 'country_Other']


### `Director` Multi-Hot

In [8]:
director_counts = (
    tmdb_df["director"]
    .str.split("|")
    .explode()
    .value_counts()
)

print("Number of unique directors:", len(director_counts))

print("\nFrequency summary:")
print(director_counts.describe())

print("\nNumber of directors with:")
print(">= 2 movies:", (director_counts >= 2).sum())
print(">= 5 movies:", (director_counts >= 5).sum())
print(">= 15 movies:", (director_counts >= 15).sum())
print(">= 20 movies:", (director_counts >= 20).sum())
print(">= 50 movies:", (director_counts >= 50).sum())

Number of unique directors: 38248

Frequency summary:
count    38248.000000
mean         2.487032
std          4.096509
min          1.000000
25%          1.000000
50%          1.000000
75%          2.000000
max        118.000000
Name: count, dtype: float64

Number of directors with:
>= 2 movies: 14622
>= 5 movies: 4239
>= 15 movies: 757
>= 20 movies: 435
>= 50 movies: 30


We will only keep Directors with movie counts >= 10, otherwise we will have too many features. Uncommon directors will be considered as others (test 15)

In [9]:
# Fill missing directors
tmdb_df["director"] = tmdb_df["director"].fillna("Unknown")

# Keep directors who appear in at least 15 (test) movies
common_directors = set(
    director_counts[director_counts >= 15].index
)

def keep_common_directors(value):
    directors = value.split("|")
    kept = [d for d in directors if d in common_directors]

    if len(kept) == 0:
        return "Other"

    return "|".join(kept)

director_grouped = tmdb_df["director"].apply(keep_common_directors)

director_features = (
    director_grouped
    .str.get_dummies(sep="|")
    .add_prefix("director_")
)

print("Number of director features:", director_features.shape[1])

Number of director features: 758


### `Top_Cast`

In [10]:
cast_counts = (
    tmdb_df["top_cast"]
    .fillna("Unknown")
    .str.split("|")
    .explode()
    .value_counts()
)

print("Number of unique cast members:", len(cast_counts))

print("\nFrequency summary:")
print(cast_counts.describe())

print("\nNumber of cast members with:")
print(">= 5 movies:", (cast_counts >= 5).sum())
print(">= 10 movies:", (cast_counts >= 10).sum())
print(">= 30 movies:", (cast_counts >= 30).sum())
print(">= 50 movies:", (cast_counts >= 50).sum())
print(">= 100 movies:", (cast_counts >= 100).sum())

Number of unique cast members: 257118

Frequency summary:
count    257118.000000
mean          2.879608
std           8.375327
min           1.000000
25%           1.000000
50%           1.000000
75%           2.000000
max        3076.000000
Name: count, dtype: float64

Number of cast members with:
>= 5 movies: 32502
>= 10 movies: 14297
>= 30 movies: 2531
>= 50 movies: 743
>= 100 movies: 39


In [11]:
# Fill missing cast
tmdb_df["top_cast"] = tmdb_df["top_cast"].fillna("Unknown")

# Keep cast members appearing in at least 30 (from 20->30 test) movies
common_cast = set(
    cast_counts[cast_counts >= 30].index
)

def keep_common_cast(value):
    cast = value.split("|")
    kept = [actor for actor in cast if actor in common_cast]

    if len(kept) == 0:
        return "Other"

    return "|".join(kept)

cast_grouped = tmdb_df["top_cast"].apply(keep_common_cast)

cast_features = (
    cast_grouped
    .str.get_dummies(sep="|")
    .add_prefix("cast_")
)

print("Number of cast features:", cast_features.shape[1])

Number of cast features: 2532


### `Keywords`

In [12]:
keyword_counts = (
    tmdb_df["keywords"]
    .fillna("Unknown")
    .str.split("|")
    .explode()
    .value_counts()
)

print("Number of unique keywords:", len(keyword_counts))

print("\nFrequency summary:")
print(keyword_counts.describe())

print("\nNumber of keywords with:")
print(">= 5 movies:", (keyword_counts >= 5).sum())
print(">= 10 movies:", (keyword_counts >= 10).sum())
print(">= 20 movies:", (keyword_counts >= 20).sum())
print(">= 50 movies:", (keyword_counts >= 50).sum())
print(">= 100 movies:", (keyword_counts >= 100).sum())

Number of unique keywords: 32686

Frequency summary:
count    32686.000000
mean        12.233709
std        142.587237
min          1.000000
25%          1.000000
50%          2.000000
75%          7.000000
max      23418.000000
Name: count, dtype: float64

Number of keywords with:
>= 5 movies: 10841
>= 10 movies: 6187
>= 20 movies: 3270
>= 50 movies: 1346
>= 100 movies: 597


In [13]:
# Keep real keywords appearing in at least 50 movies (20-> 50 test)
common_keywords = set(
    keyword_counts[
        (keyword_counts >= 50) &
        (keyword_counts.index != "Unknown")
    ].index
)

def keep_common_keywords(value):
    if pd.isna(value):
        return "Other"

    keywords = value.split("|")
    kept = [k for k in keywords if k in common_keywords]

    if len(kept) == 0:
        return "Other"

    return "|".join(kept)

keyword_grouped = tmdb_df["keywords"].apply(keep_common_keywords)

keyword_features = (
    keyword_grouped
    .str.get_dummies(sep="|")
    .add_prefix("keyword_")
)

print("Number of keyword features:", keyword_features.shape[1])

Number of keyword features: 1346


### `Overview` NLP Text Embedding

In [14]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")



c:\Users\asus\miniconda3\envs\ml_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6491.85it/s]


In [15]:
all_overviews = (
    tmdb_df["overview"]
    .fillna("")
    .tolist()
)

overview_embeddings = embedding_model.encode(
    all_overviews,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Overview embedding shape:", overview_embeddings.shape)

Batches: 100%|██████████| 1348/1348 [09:35<00:00,  2.34it/s]


Overview embedding shape: (86250, 384)


### `Production_company`

In [16]:
company_counts = (
    tmdb_df["production_companies"]
    .fillna("Unknown")
    .str.split("|")
    .explode()
    .value_counts()
)

print("Number of unique production companies:", len(company_counts))

print("\nFrequency summary:")
print(company_counts.describe())

print("\nNumber of production companies with:")
print(">= 5 movies:", (company_counts >= 5).sum())
print(">= 10 movies:", (company_counts >= 10).sum())
print(">= 20 movies:", (company_counts >= 20).sum())
print(">= 50 movies:", (company_counts >= 50).sum())
print(">= 100 movies:", (company_counts >= 100).sum())

Number of unique production companies: 48451

Frequency summary:
count    48451.000000
mean         3.861860
std         49.455967
min          1.000000
25%          1.000000
50%          1.000000
75%          2.000000
max       9989.000000
Name: count, dtype: float64

Number of production companies with:
>= 5 movies: 6377
>= 10 movies: 2808
>= 20 movies: 1180
>= 50 movies: 355
>= 100 movies: 137


In [17]:
# Keep real production companies appearing in at least 50 movies (20->50 test)
common_companies = set(
    company_counts[
        (company_counts >= 50) &
        (company_counts.index != "Unknown")
    ].index
)

def keep_common_companies(value):
    if pd.isna(value):
        return "Other"

    companies = value.split("|")
    kept = [c for c in companies if c in common_companies]

    if len(kept) == 0:
        return "Other"

    return "|".join(kept)

company_grouped = tmdb_df["production_companies"].apply(
    keep_common_companies
)

company_features = (
    company_grouped
    .str.get_dummies(sep="|")
    .add_prefix("company_")
)

print("Number of production company features:", company_features.shape[1])

Number of production company features: 355


### Feature Engineering Summary

| Feature | Method | Dimensions |
|---|---|---:|
| Genre | Multi-hot encoding | 19 |
| Runtime | StandardScaler | 1 |
| Original Language | Rare languages (<100 movies) → Other + One-hot encoding | 35 |
| Origin Country | Rare countries (<100 movies) → Other + Multi-hot encoding | 45 |
| Director | Keep directors with ≥10 movies + Multi-hot encoding | 1,501 |
| Top Cast | Keep cast members with ≥20 movies + Multi-hot encoding | 5,261 |
| Keywords | Keep keywords appearing in ≥20 movies + Multi-hot encoding | 3,270 |
| Overview | NLP Sentence Transformer (`all-MiniLM-L6-v2`) | 384 |
| Production Companies | Keep companies with ≥20 movies + Multi-hot encoding | 1,180 |
| **Total** | | **11,696** |

**Note:** Runtime is currently standardized using the full TMDB dataset. After merging with the MovieLens training and test data, the scaler should be refitted using the training data only to avoid data leakage.

 ### **Since we have too many features right now, we will need to do some dimension reduction**

# Dimensionality Reduction — Truncated SVD

In [18]:
# Combine all high-dimensional features into a single sparse matrix
from scipy.sparse import csr_matrix, hstack

high_dim_features = hstack([
    csr_matrix(director_features.values),
    csr_matrix(cast_features.values),
    csr_matrix(keyword_features.values),
    csr_matrix(company_features.values)
])

print("High-dimensional feature shape:", high_dim_features.shape)

High-dimensional feature shape: (86250, 4991)


In [19]:
from sklearn.decomposition import TruncatedSVD
import numpy as np

svd_test = TruncatedSVD(
    n_components=150,
    random_state=42
)

svd_test.fit(high_dim_features)

cumulative_variance = np.cumsum(
    svd_test.explained_variance_ratio_
)

for n in [50, 100, 150]:
    print(
        f"{n} components: "
        f"{cumulative_variance[n-1]:.2%} explained variance"
    )

#try 300 -> 150

50 components: 29.04% explained variance
100 components: 34.46% explained variance
150 components: 38.13% explained variance


In [20]:
high_dim_svd = svd_test.transform(high_dim_features)

print("Reduced feature shape:", high_dim_svd.shape)
print(
    "Explained variance:",
    f"{svd_test.explained_variance_ratio_.sum():.2%}"
)

Reduced feature shape: (86250, 150)
Explained variance: 38.13%


## Combine Movie Features

In [21]:
final_movie_features = np.hstack([
    #genre_features.values,  #remove duplicate of genre features
    runtime_feature,
    language_features.values,
    country_features.values,
    overview_embeddings,
    high_dim_svd
])

print("Final movie feature shape:", final_movie_features.shape)

Final movie feature shape: (86250, 561)


## Save

In [22]:
import os
import numpy as np

output_path = "../data/tmdb_movie_features.npz"

np.savez_compressed(
    output_path,
    movie_ids=tmdb_df["movieId"].to_numpy(),
    features=final_movie_features
)

size_mb = os.path.getsize(output_path) / (1024 ** 2)

print(f"File size: {size_mb:.2f} MB")

File size: 232.08 MB
